# 07 — Timestral d+5

O [Timestral](https://github.com/mistralai/timestral) (Mistral) no mesmo corte do 04/05:

1. **Contexto** = caudal horário **antes de 1/10/2026** (série do 04).
2. **Teste** = caudal **a partir de 1/10/2026** (descargas NIVUS do 05), sem alterar o caudal medido.
3. Em cada âncora o modelo prevê **d+5** (120 h). Vê o histórico do 04 e o caudal do 05 já medido até essa hora; **não** vê o caudal futuro.
4. A chuva do horizonte entra só como covariável (`known_covariates`): a que realmente caiu, arquivo Open-Meteo.
5. Métricas só nas horas com caudal medido. A afluência direta sai do `split_flow`. No Ameal, horas sem velocidade ficam de fora.

A biblioteca pública é só inferência: os pesos são os da Mistral. Não há fine-tune local nem árvore ao lado.

O Timestral instala-se com o projeto (`uv add git+https://github.com/mistralai/timestral.git`). Se o kernel não vir o pacote, reinicia e corre de novo.


## 1. Setup

Imports, corte 1/10/2026 e o modelo Timestral.


In [1]:
import sys
from dataclasses import replace
from pathlib import Path

cwd = Path.cwd().resolve()
predict_dir = cwd if (cwd / "previsao.py").exists() else cwd / "predict inflow"
project_dir = predict_dir.parent
for path in (project_dir, predict_dir):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import requests
from plotly.subplots import make_subplots
from sklearn.metrics import r2_score
from timestral import TimestralForecaster

from pipeline_config import OPEN_METEO_ARCHIVE_URL, OPEN_METEO_TIMEZONE, STATIONS
from previsao import (
    HORIZONTE_DIAS,
    MIN_LEITURAS_HORA,
    OUTPUT_DIR,
    componentes,
    hourly_flow,
    load_flow,
    load_precip,
    split_flow,
)

MODELO = "mistralai/Timestral-28M"
CORTE = pd.Timestamp("2026-10-01")
HORAS_D5 = HORIZONTE_DIAS * 24
DADOS_NOVOS = Path.home() / "Downloads" / "Dados de Projeto" / "dados-novos-pra-testar-no-modelo"
L_S_TO_M3_H = 3.6
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

forecaster = TimestralForecaster.from_pretrained(MODELO)
print("modelo:", MODELO, "| corte:", f"{CORTE:%d/%m/%Y}", "| horizonte:", HORAS_D5, "h")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

modelo: mistralai/Timestral-28M | corte: 01/10/2026 | horizonte: 120 h


## 2. Contexto (antes de 1/10/2026)

Caudal horário do 04, só horas **< 1/10/2026**. É o histórico com que o Timestral se condiciona.

As constantes do `split_flow` também saem daqui (no Antanhol, no teste, usam-se as do notebook 06). Servem só para as métricas da afluência direta — o Timestral prevê o caudal total.


In [2]:
contexto_estacao = {}
for estacao in STATIONS:
    precip = load_precip(estacao)
    flow = load_flow(estacao)
    flow = flow[flow["datetime"] < CORTE]
    hourly = hourly_flow(flow)
    hourly = hourly[hourly.index < CORTE]
    caudal = hourly["caudal_total_m3_h"].copy()
    caudal = caudal.reindex(pd.date_range(caudal.index.min(), caudal.index.max(), freq="h"))
    caudal.index.name = "datetime"
    comp = componentes(flow, precip)
    if estacao == "antanhol":
        comp_teste = replace(comp, dias_antes=1, indireta_m3_h=1.188, esgoto_m3_h=15.059727)
    else:
        comp_teste = comp
    contexto_estacao[estacao] = {
        "caudal": caudal,
        "comp_teste": comp_teste,
        "precip": precip,
    }
    print(
        f"{STATIONS[estacao]['folder']}: {caudal.notna().sum()} horas de contexto, "
        f"{caudal.index.min():%d/%m} → {caudal.index.max():%d/%m}"
    )


Ameal: 302 horas de contexto, 22/07 → 02/09
Antanhol: 1006 horas de contexto, 22/07 → 02/09


## 3. Teste (a partir de 1/10/2026)

Descargas NIVUS do 05, tal como estão no CSV. Só horas **≥ 1/10/2026** com caudal medido (≥15 leituras válidas na hora).

A chuva do arquivo Open-Meteo entra mais abaixo como covariável do Timestral, não nos gráficos.


In [3]:
COLUNAS = {"caudal_l_s": "app1_q", "altura_m": "app1_h", "velocidade_m_s": "app1_v"}


def read_nivus(path: Path) -> pd.DataFrame:
    with path.open(encoding="utf-8-sig") as handle:
        header_row = next(i for i, line in enumerate(handle) if line.lower().startswith("date;time"))
    raw = pd.read_csv(path, sep=";", skiprows=header_row, encoding="utf-8-sig", dtype=str).dropna(subset=["Date", "Time"])
    frame = pd.DataFrame(
        {"datetime": pd.to_datetime(raw["Date"] + " " + raw["Time"], format="%d.%m.%Y %H:%M:%S", errors="coerce")}
    )
    for name, fragment in COLUNAS.items():
        column = next(c for c in raw.columns if fragment in c.lower())
        frame[name] = pd.to_numeric(raw[column].str.replace(",", ".", regex=False), errors="coerce")
    frame["caudal_m3_h"] = frame["caudal_l_s"] * L_S_TO_M3_H
    return frame


def load_new(estacao: str) -> pd.DataFrame:
    prefix = STATIONS[estacao]["folder"].lower()
    files = [
        f for folder in DADOS_NOVOS.iterdir() if folder.is_dir() and folder.name.lower().startswith(prefix)
        for f in sorted(folder.rglob("*"))
        if f.is_file() and f.parent.name.upper() == "DATA" and f.suffix.lower() == ".csv"
    ]
    if not files:
        raise FileNotFoundError(f"Nenhum CSV em DATA/ para {estacao} em {DADOS_NOVOS}")
    frame = (
        pd.concat([read_nivus(f) for f in files], ignore_index=True)
        .dropna(subset=["datetime"])
        .drop_duplicates("datetime", keep="last")
        .sort_values("datetime")
        .reset_index(drop=True)
    )
    diameter = float(STATIONS[estacao]["collector_diameter_m"])
    frame["altura_valida"] = (frame["altura_m"] > 0) & (frame["altura_m"] < diameter)
    frame["caudal_valido"] = frame["altura_valida"] & (frame["caudal_l_s"] > 0)
    return frame


def hourly_teste(flow: pd.DataFrame) -> pd.Series:
    """Caudal médio da hora, só com pelo menos 15 leituras de caudal válido."""
    flow = flow.assign(hora=flow["datetime"].dt.floor("h"))
    validas = flow[flow["caudal_valido"]]
    grouped = validas.groupby("hora")["caudal_m3_h"].agg(caudal="mean", n="size")
    series = grouped.loc[grouped["n"] >= MIN_LEITURAS_HORA, "caudal"]
    series.index.name = "datetime"
    series.name = "medido"
    return series


def fetch_rain(estacao: str, inicio: pd.Timestamp, fim: pd.Timestamp) -> pd.Series:
    station = STATIONS[estacao]
    response = requests.get(
        OPEN_METEO_ARCHIVE_URL,
        params={
            "latitude": station["latitude"],
            "longitude": station["longitude"],
            "hourly": "precipitation",
            "timezone": OPEN_METEO_TIMEZONE,
            "start_date": f"{inicio:%Y-%m-%d}",
            "end_date": f"{fim:%Y-%m-%d}",
        },
        timeout=30,
    )
    response.raise_for_status()
    hourly = response.json()["hourly"]
    series = pd.Series(hourly["precipitation"], index=pd.to_datetime(hourly["time"]), name="precip_mm", dtype=float)
    series.index.name = "datetime"
    return series.fillna(0.0)


teste = {}
for estacao in STATIONS:
    flow = load_new(estacao)
    medido = hourly_teste(flow)
    medido = medido[medido.index >= CORTE]
    chuva = fetch_rain(
        estacao,
        contexto_estacao[estacao]["caudal"].index.min(),
        medido.index.max() + pd.Timedelta(days=HORIZONTE_DIAS),
    )
    teste[estacao] = {"medido": medido, "chuva": chuva}
    print(
        f"{STATIONS[estacao]['folder']}: {len(medido)} horas com caudal medido "
        f"({medido.index.min():%d/%m %H:%M} → {medido.index.max():%d/%m %H:%M})"
    )


Ameal: 108 horas com caudal medido (01/10 10:00 → 06/10 00:00)
Antanhol: 305 horas com caudal medido (01/10 11:00 → 14/10 09:00)


## 4. Previsão d+5 com o Timestral

Âncoras: 30/09 23:00 (véspera do corte) e o último caudal medido de cada dia ≥ 1/10.

Em cada âncora o Timestral recebe o caudal até essa hora e a chuva conhecida no horizonte, e devolve 120 horas. Só se guarda o que cai no período de teste (≥ 1/10).


In [4]:
def contexto(estacao: str, origem: pd.Timestamp) -> pd.Series:
    """caudais iguais ao do notebook 04 pra colocar no forecaster"""
    passado = pd.concat([contexto_estacao[estacao]["caudal"], teste[estacao]["medido"]])
    passado = passado[~passado.index.duplicated(keep="last")].sort_index()
    passado = passado.loc[passado.index <= origem]
    return passado.reindex(pd.date_range(passado.index.min(), origem, freq="h"))


def chuva_conhecida(estacao: str, context_index: pd.DatetimeIndex, horizonte: pd.DatetimeIndex) -> pd.Series:
    full = context_index.append(horizonte)
    hist = contexto_estacao[estacao]["precip"].reindex(full)
    novas = teste[estacao]["chuva"].reindex(full)
    rain = hist.fillna(novas).fillna(0.0)
    rain.name = "precip_mm"
    return rain


def ancora_d5(estacao: str, origem: pd.Timestamp) -> pd.DataFrame:
    ctx = contexto(estacao, origem)
    horizonte = pd.date_range(origem + pd.Timedelta(hours=1), periods=HORAS_D5, freq="h")
    rain = chuva_conhecida(estacao, ctx.index, horizonte)
    pred = forecaster.forecast(
        ctx.to_numpy(dtype=float)[None, None, :],
        prediction_length=HORAS_D5,
        known_covariates=rain.to_numpy(dtype=float)[None, None, :],
    )
    timestral = pd.Series(np.asarray(pred.median)[0, 0], index=horizonte).clip(lower=0.0)
    medido = teste[estacao]["medido"].reindex(horizonte)
    lead = (horizonte.normalize() - origem.normalize()).days
    return pd.DataFrame({
        "datetime": horizonte,
        "origin": origem,
        "lead_dias": lead,
        "medido": medido.to_numpy(),
        "timestral": timestral.to_numpy(),
    })


def origens(estacao: str) -> list[pd.Timestamp]:
    stamps = {CORTE - pd.Timedelta(hours=1)}
    medido = teste[estacao]["medido"]
    for _, group in medido.groupby(medido.index.date):
        stamps.add(pd.Timestamp(group.index.max()))
    return sorted(s for s in stamps if s < medido.index.max())


detalhe = {}
for estacao in STATIONS:
    pecas = [ancora_d5(estacao, origem) for origem in origens(estacao)]
    frame = pd.concat(pecas, ignore_index=True)
    frame = frame[frame["datetime"] >= CORTE]
    detalhe[estacao] = frame
    path = OUTPUT_DIR / f"{estacao}_timestral_04a05.csv"
    frame.to_csv(path, index=False)
    print(f"{STATIONS[estacao]['folder']}: {len(pecas)} âncoras, {path.name}")


/Users/marcosgrandesso/wada/afluencia_indevida_pipeline/.venv/lib/python3.12/site-packages/timestral/forecaster.py:234: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  x = torch.as_tensor(context, dtype=torch.float32, device=self.device)


Ameal: 6 âncoras, ameal_timestral_04a05.csv
Antanhol: 14 âncoras, antanhol_timestral_04a05.csv


## 5. Métricas

R² e MAE do Timestral no **caudal total** e na **afluência direta**, só nas horas com NIVUS a partir de 1/10.


In [5]:
def metricas(medido: pd.Series, previsto: pd.Series, comp, when: pd.Series | None = None) -> pd.Series:
    par = pd.concat({"medido": medido, "previsto": previsto}, axis=1).dropna()
    if par.empty:
        return pd.Series({"n": 0, "mae_caudal": np.nan, "r2_caudal": np.nan, "mae_direta": np.nan, "r2_direta": np.nan})
    if when is not None:
        par = par.copy()
        par.index = pd.to_datetime(when.loc[par.index])
    d_obs = split_flow(par["medido"], comp)["direta_m3_h"]
    d_hat = split_flow(par["previsto"], comp)["direta_m3_h"]
    return pd.Series({
        "n": int(len(par)),
        "mae_caudal": float((par["previsto"] - par["medido"]).abs().mean()),
        "r2_caudal": float(r2_score(par["medido"], par["previsto"])),
        "mae_direta": float((d_hat - d_obs).abs().mean()),
        "r2_direta": float(r2_score(d_obs, d_hat)),
    })


resumo_linhas = []
for estacao in STATIONS:
    frame = detalhe[estacao].dropna(subset=["medido"])
    comp = contexto_estacao[estacao]["comp_teste"]
    m = metricas(frame["medido"], frame["timestral"], comp, frame["datetime"])
    print(f"\n=== {STATIONS[estacao]['folder']} — {int(m.n)} pares âncora/hora com caudal medido ===")
    print(
        f"  caudal  R² {m.r2_caudal:.3f} · MAE {m.mae_caudal:.2f} m³/h | "
        f"direta R² {m.r2_direta:.3f} · MAE {m.mae_direta:.2f} m³/h"
    )
    print("  por lead (Timestral):")
    for lead, grupo in frame.groupby("lead_dias"):
        if 1 <= lead <= HORIZONTE_DIAS:
            ml = metricas(grupo["medido"], grupo["timestral"], comp, grupo["datetime"])
            print(
                f"    d+{int(lead)}: caudal R² {ml.r2_caudal:.3f} · MAE {ml.mae_caudal:.2f} | "
                f"direta R² {ml.r2_direta:.3f} · MAE {ml.mae_direta:.2f} ({int(ml.n)} h)"
            )
    resumo_linhas.append({"estacao": estacao, "modelo": "Timestral", **m.to_dict()})

resumo = pd.DataFrame(resumo_linhas)
resumo.to_csv(OUTPUT_DIR / "07_timestral_metricas.csv", index=False)
resumo



=== Ameal — 340 pares âncora/hora com caudal medido ===
  caudal  R² 0.086 · MAE 0.97 m³/h | direta R² 0.007 · MAE 0.80 m³/h
  por lead (Timestral):
    d+1: caudal R² 0.158 · MAE 0.80 | direta R² 0.059 · MAE 0.64 (108 h)
    d+2: caudal R² 0.146 · MAE 0.87 | direta R² 0.045 · MAE 0.69 (94 h)
    d+3: caudal R² 0.048 · MAE 1.02 | direta R² -0.028 · MAE 0.83 (70 h)
    d+4: caudal R² 0.013 · MAE 1.17 | direta R² -0.043 · MAE 1.04 (46 h)
    d+5: caudal R² -0.033 · MAE 1.62 | direta R² -0.069 · MAE 1.48 (22 h)

=== Antanhol — 1329 pares âncora/hora com caudal medido ===
  caudal  R² -0.156 · MAE 8.08 m³/h | direta R² -0.502 · MAE 3.67 m³/h
  por lead (Timestral):
    d+1: caudal R² 0.418 · MAE 5.21 | direta R² 0.133 · MAE 2.32 (305 h)
    d+2: caudal R² 0.173 · MAE 6.97 | direta R² -0.223 · MAE 3.07 (292 h)
    d+3: caudal R² -0.203 · MAE 8.69 | direta R² -0.585 · MAE 3.85 (268 h)
    d+4: caudal R² -0.652 · MAE 10.13 | direta R² -0.933 · MAE 4.62 (244 h)
    d+5: caudal R² -0.656 · MAE

,estacao,modelo,n,mae_caudal,r2_caudal,mae_direta,r2_direta
0,ameal,Timestral,340.0,0.965805,0.086276,0.804105,0.007129
1,antanhol,Timestral,1329.0,8.083285,-0.155565,3.670231,-0.501589


# Daqui para baixo é só plotar gráficos

A previsão já está feita. Estas células só desenham o **Medido** e o **Timestral**. A legenda leva o R² e o MAE do caudal total e da afluência direta.


## 6. Gráficos (Medido vs Timestral)


In [6]:
def figura(estacao: str) -> go.Figure:
    frame = detalhe[estacao]
    medido = teste[estacao]["medido"]
    por_hora = frame.dropna(subset=["timestral"]).groupby("datetime")["timestral"].mean()
    comp = contexto_estacao[estacao]["comp_teste"]
    m = metricas(frame["medido"], frame["timestral"], comp, frame["datetime"])
    nome_caudal = f"Timestral  caudal R² {m.r2_caudal:.3f} · MAE {m.mae_caudal:.2f}"
    nome_direta = f"Timestral  direta R² {m.r2_direta:.3f} · MAE {m.mae_direta:.2f}"
    label = STATIONS[estacao]["folder"]
    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True, row_heights=[0.5, 0.5], vertical_spacing=0.08,
        subplot_titles=("Caudal total (m³/h)", "Afluência direta (m³/h)"),
    )
    fig.add_trace(
        go.Scatter(
            x=medido.index, y=medido, name="Medido", mode="lines",
            line=dict(color="navy", width=1.5), legendgroup="medido",
        ),
        row=1, col=1,
    )
    fig.add_trace(
        go.Scatter(x=por_hora.index, y=por_hora, name=nome_caudal, mode="lines", line=dict(color="red", width=2)),
        row=1, col=1,
    )
    d_obs = split_flow(medido, comp)["direta_m3_h"]
    d_ts = split_flow(por_hora, comp)["direta_m3_h"]
    fig.add_trace(
        go.Scatter(
            x=d_obs.index, y=d_obs, name="Medido", mode="lines",
            line=dict(color="navy", width=1.5), legendgroup="medido", showlegend=False,
        ),
        row=2, col=1,
    )
    fig.add_trace(
        go.Scatter(x=d_ts.index, y=d_ts, name=nome_direta, mode="lines", line=dict(color="red", width=2)),
        row=2, col=1,
    )
    fig.update_traces(yhoverformat=".2f")
    fig.update_layout(
        title_text=f"Timestral d+5 — {label} (teste ≥ 1/10)",
        height=720, template="plotly_white", hovermode="x unified",
        hoverlabel=dict(namelength=-1),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
    )
    return fig


for estacao in STATIONS:
    fig = figura(estacao)
    html_path = OUTPUT_DIR / f"07_timestral_{estacao}.html"
    fig.write_html(html_path)
    print(html_path)
    fig.show()


/Users/marcosgrandesso/wada/afluencia_indevida_pipeline/predict inflow/output/07_timestral_ameal.html


/Users/marcosgrandesso/wada/afluencia_indevida_pipeline/predict inflow/output/07_timestral_antanhol.html
